In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

# Define a CNN model (≤ 10 layers, ≤ 8M params)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)  # Adapted for different class splits

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# Load CIFAR-100 dataset (modify for EMNIST/Fashion MNIST if needed)
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
testset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

# Select specific class indices for each model
class_indices = [
    [0, 1, 2, 3, 4],  # Model 1 classes
    [5, 6, 7, 8, 9],  # Model 2 classes
    [10, 11, 12, 13, 14]  # Model 3 classes
]

# Create datasets per model
def filter_dataset(dataset, class_ids):
    indices = [i for i, (_, label) in enumerate(dataset) if label in class_ids]
    subset = torch.utils.data.Subset(dataset, indices)
    return subset

train_loaders = [DataLoader(filter_dataset(trainset, ids), batch_size=64, shuffle=True) for ids in class_indices]
test_loaders = [DataLoader(filter_dataset(testset, ids), batch_size=64, shuffle=False) for ids in class_indices]

# Training function
def train_model(model, train_loader, epochs=10, lr=0.001):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    print(f"Model Trained for {epochs} epochs")
    return model

# Train three separate models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models = [SimpleCNN(num_classes=5).to(device) for _ in range(3)]
trained_models = [train_model(models[i], train_loaders[i]) for i in range(3)]

# Save models
for i, model in enumerate(trained_models):
    torch.save(model.state_dict(), f"model_{i+1}.pth")


Files already downloaded and verified
Files already downloaded and verified
Model Trained for 10 epochs


IndexError: Target 5 is out of bounds.

In [ ]:
import torch.nn.functional as F

def ensemble_predict(models, input_tensor):
    models = [model.eval() for model in models]  # Set all models to eval mode
    predictions = [F.softmax(model(input_tensor), dim=1) for model in models]
    avg_prediction = torch.mean(torch.stack(predictions), dim=0)
    return torch.argmax(avg_prediction, dim=1)  # Final class prediction

# Load trained models
models = [SimpleCNN(num_classes=5).to(device) for _ in range(3)]
for i in range(3):
    models[i].load_state_dict(torch.load(f"model_{i+1}.pth"))
    models[i].eval()

# Test the ensemble model
sample_input = torch.randn(1, 3, 32, 32).to(device)  # Random input
prediction = ensemble_predict(models, sample_input)
print("Final Prediction:", prediction)


In [ ]:
class mergedCNN(nn.Module):
    def __init__(self, models):
        super(mergedCNN, self).__init__()
        self.feature_extractors = [nn.Sequential(*list(model.children())[:-1]) for model in models]  # Remove final FC layers
        self.fc = nn.Linear(3 * 256, 15)  # Merge feature vectors into 15-class output

    def forward(self, x):
        features = [extractor(x).view(x.size(0), -1) for extractor in self.feature_extractors]
        merged_features = torch.cat(features, dim=1)  # Concatenate feature vectors
        return self.fc(merged_features)

# Create merged model
merged_model = mergedCNN(models).to(device)


In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

# Test accuracy for ensemble
test_accuracy = evaluate_model(merged_model, test_loaders[0])  # Modify for full dataset
print(f"Unified Model Test Accuracy: {test_accuracy:.2%}")
